In [1]:
import re
import nltk
import spacy

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
movies = [
    {"title": "Galactic Frontier",
     "description": "An astronaut and her robot companion explore uncharted planets in a distant galaxy, battling alien creatures and discovering ancient civilizations."},
    {"title": "The Last Harvest",
     "description": "A farmer struggles to save his organic crops during a devastating drought, learning sustainable agricultural techniques passed down through generations."},
    {"title": "Iron Circuit",
     "description": "In a futuristic city, a rogue AI robot gains consciousness and teams up with a human engineer to fight against corporate machine overlords."},
    {"title": "Ocean's Breath",
     "description": "A marine biologist dives deep into unexplored ocean trenches, discovering bioluminescent creatures and fighting to protect the underwater ecosystem."},
    {"title": "The Noodle Master",
     "description": "A young chef travels across Asia learning ancient recipes, blending traditional cooking methods with healthy, fresh ingredients to win a culinary competition."},
    {"title": "Red Planet Rush",
     "description": "A crew of astronauts races against time to establish the first human colony on Mars, facing dust storms, radiation, and equipment failures in space."},
    {"title": "Greenplate",
     "description": "A nutritionist and a celebrity chef team up to revolutionize school cafeterias by replacing junk food with delicious, healthy, plant-based meals."},
    {"title": "Steel Horizon",
     "description": "Giant battle robots controlled by pilots clash in an apocalyptic wasteland, where one pilot questions whether the endless war is worth fighting."},
    {"title": "Whispers of the Forest",
     "description": "A botanist lives alone in a rainforest, studying rare medicinal plants, and must defend the ecosystem from illegal logging companies."},
    {"title": "Cosmos Detective",
     "description": "A sharp-witted detective solves crimes aboard a massive space station orbiting Saturn, using science and logic to catch interstellar criminals."},
]

# combine title + description into one string per movie
raw_docs = [m["title"] + " " + m["description"] for m in movies]

print("Total movies:", len(movies))
print("Sample raw doc:")
print(raw_docs[0])


Total movies: 10
Sample raw doc:
Galactic Frontier An astronaut and her robot companion explore uncharted planets in a distant galaxy, battling alien creatures and discovering ancient civilizations.


In [3]:
# remove punctuation
docs_no_punct = [re.sub(r'[^\w\s]', '', doc) for doc in raw_docs]

# remove extra whitespace
docs_clean = [re.sub(r'\s+', ' ', doc).strip() for doc in docs_no_punct]

print("\nCleaned Text :", docs_clean)



Cleaned Text : ['Galactic Frontier An astronaut and her robot companion explore uncharted planets in a distant galaxy battling alien creatures and discovering ancient civilizations', 'The Last Harvest A farmer struggles to save his organic crops during a devastating drought learning sustainable agricultural techniques passed down through generations', 'Iron Circuit In a futuristic city a rogue AI robot gains consciousness and teams up with a human engineer to fight against corporate machine overlords', 'Oceans Breath A marine biologist dives deep into unexplored ocean trenches discovering bioluminescent creatures and fighting to protect the underwater ecosystem', 'The Noodle Master A young chef travels across Asia learning ancient recipes blending traditional cooking methods with healthy fresh ingredients to win a culinary competition', 'Red Planet Rush A crew of astronauts races against time to establish the first human colony on Mars facing dust storms radiation and equipment failur

In [4]:
# lowercase
docs_lower = [doc.lower() for doc in docs_clean]

print("\nLowercase Text:", docs_lower)



Lowercase Text: ['galactic frontier an astronaut and her robot companion explore uncharted planets in a distant galaxy battling alien creatures and discovering ancient civilizations', 'the last harvest a farmer struggles to save his organic crops during a devastating drought learning sustainable agricultural techniques passed down through generations', 'iron circuit in a futuristic city a rogue ai robot gains consciousness and teams up with a human engineer to fight against corporate machine overlords', 'oceans breath a marine biologist dives deep into unexplored ocean trenches discovering bioluminescent creatures and fighting to protect the underwater ecosystem', 'the noodle master a young chef travels across asia learning ancient recipes blending traditional cooking methods with healthy fresh ingredients to win a culinary competition', 'red planet rush a crew of astronauts races against time to establish the first human colony on mars facing dust storms radiation and equipment failu

In [5]:
# tokenization
docs_tokens = [word_tokenize(doc) for doc in docs_lower]

print("\nTokens:", docs_tokens)



Tokens: [['galactic', 'frontier', 'an', 'astronaut', 'and', 'her', 'robot', 'companion', 'explore', 'uncharted', 'planets', 'in', 'a', 'distant', 'galaxy', 'battling', 'alien', 'creatures', 'and', 'discovering', 'ancient', 'civilizations'], ['the', 'last', 'harvest', 'a', 'farmer', 'struggles', 'to', 'save', 'his', 'organic', 'crops', 'during', 'a', 'devastating', 'drought', 'learning', 'sustainable', 'agricultural', 'techniques', 'passed', 'down', 'through', 'generations'], ['iron', 'circuit', 'in', 'a', 'futuristic', 'city', 'a', 'rogue', 'ai', 'robot', 'gains', 'consciousness', 'and', 'teams', 'up', 'with', 'a', 'human', 'engineer', 'to', 'fight', 'against', 'corporate', 'machine', 'overlords'], ['oceans', 'breath', 'a', 'marine', 'biologist', 'dives', 'deep', 'into', 'unexplored', 'ocean', 'trenches', 'discovering', 'bioluminescent', 'creatures', 'and', 'fighting', 'to', 'protect', 'the', 'underwater', 'ecosystem'], ['the', 'noodle', 'master', 'a', 'young', 'chef', 'travels', 'acr

In [6]:
# remove stop words
# keeping 'against' because they carry meaning in descriptions
sentiments = { "against"}

stop_words = set(stopwords.words('english'))

docs_filtered = [
    [token for token in tokens if token not in stop_words or token in sentiments]
    for tokens in docs_tokens
]

print("\nFiltered Tokens :", docs_filtered)



Filtered Tokens : [['galactic', 'frontier', 'astronaut', 'robot', 'companion', 'explore', 'uncharted', 'planets', 'distant', 'galaxy', 'battling', 'alien', 'creatures', 'discovering', 'ancient', 'civilizations'], ['last', 'harvest', 'farmer', 'struggles', 'save', 'organic', 'crops', 'devastating', 'drought', 'learning', 'sustainable', 'agricultural', 'techniques', 'passed', 'generations'], ['iron', 'circuit', 'futuristic', 'city', 'rogue', 'ai', 'robot', 'gains', 'consciousness', 'teams', 'human', 'engineer', 'fight', 'against', 'corporate', 'machine', 'overlords'], ['oceans', 'breath', 'marine', 'biologist', 'dives', 'deep', 'unexplored', 'ocean', 'trenches', 'discovering', 'bioluminescent', 'creatures', 'fighting', 'protect', 'underwater', 'ecosystem'], ['noodle', 'master', 'young', 'chef', 'travels', 'across', 'asia', 'learning', 'ancient', 'recipes', 'blending', 'traditional', 'cooking', 'methods', 'healthy', 'fresh', 'ingredients', 'win', 'culinary', 'competition'], ['red', 'plan

In [7]:
# using spacy for lemmatization
nlp = spacy.load("en_core_web_sm")

docs_lemmatized_spacy = [
    [nlp(token)[0].lemma_ for token in tokens]
    for tokens in docs_filtered
]

print("\nSpacy Lemmatized Tokens :", docs_lemmatized_spacy)



Spacy Lemmatized Tokens : [['galactic', 'frontier', 'astronaut', 'robot', 'companion', 'explore', 'uncharted', 'planet', 'distant', 'galaxy', 'battle', 'alien', 'creature', 'discover', 'ancient', 'civilization'], ['last', 'harvest', 'farmer', 'struggle', 'save', 'organic', 'crop', 'devastate', 'drought', 'learn', 'sustainable', 'agricultural', 'technique', 'pass', 'generation'], ['iron', 'circuit', 'futuristic', 'city', 'rogue', 'ai', 'robot', 'gain', 'consciousness', 'team', 'human', 'engineer', 'fight', 'against', 'corporate', 'machine', 'overlord'], ['ocean', 'breath', 'marine', 'biologist', 'dive', 'deep', 'unexplored', 'ocean', 'trench', 'discover', 'bioluminescent', 'creature', 'fight', 'protect', 'underwater', 'ecosystem'], ['noodle', 'master', 'young', 'chef', 'travel', 'across', 'asia', 'learn', 'ancient', 'recipe', 'blend', 'traditional', 'cook', 'method', 'healthy', 'fresh', 'ingredient', 'win', 'culinary', 'competition'], ['red', 'planet', 'rush', 'crew', 'astronaut', 'rac

In [8]:
# rejoin tokens into strings
docs_final = [' '.join(tokens) for tokens in docs_lemmatized_spacy]

print("\nFinal cleaned string:")
print(docs_final)



Final cleaned string:
['galactic frontier astronaut robot companion explore uncharted planet distant galaxy battle alien creature discover ancient civilization', 'last harvest farmer struggle save organic crop devastate drought learn sustainable agricultural technique pass generation', 'iron circuit futuristic city rogue ai robot gain consciousness team human engineer fight against corporate machine overlord', 'ocean breath marine biologist dive deep unexplored ocean trench discover bioluminescent creature fight protect underwater ecosystem', 'noodle master young chef travel across asia learn ancient recipe blend traditional cook method healthy fresh ingredient win culinary competition', 'red planet rush crew astronaut race against time establish first human colony mar face dust storm radiation equipment failure space', 'greenplate nutritionist celebrity chef team revolutionize school cafeterias replace junk food delicious healthy plantbase meal', 'steel horizon giant battle robot con

In [9]:
# build TF-IDF matrix
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(docs_final)

print(f"\nVocabulary size: {len(vectorizer.vocabulary_)} unique words")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} documents  x  {tfidf_matrix.shape[1]} words")



Vocabulary size: 150 unique words
TF-IDF matrix shape: (10, 150)
  → 10 documents  x  150 words


In [10]:
user_query = "astronaut space planet"
query_vector = vectorizer.transform([user_query])

# cosine similarity
similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

print("\nSimilarity scores:")
print("-" * 45)
for movie, score in zip(movies, similarity_scores):

    print(f"  {movie['title']:25s}  {score:.4f} ")



Similarity scores:
---------------------------------------------
  Galactic Frontier          0.2634 
  The Last Harvest           0.0000 
  Iron Circuit               0.0000 
  Ocean's Breath             0.0000 
  The Noodle Master          0.0000 
  Red Planet Rush            0.3413 
  Greenplate                 0.0000 
  Steel Horizon              0.0000 
  Whispers of the Forest     0.0000 
  Cosmos Detective           0.1105 


In [11]:
# top 3 results
top3_indices = similarity_scores.argsort()[::-1][:3]

print(f'\nQuery: "{user_query}"')

print( "\nTOP 3 RECOMMENDATIONS :")

for rank, idx in enumerate(top3_indices, start=1):
    print(f"\n{rank}  {movies[idx]['title']}")
    print(f"    {movies[idx]['description']}")
    print(f"    Similarity Score: {similarity_scores[idx]:.4f}")



Query: "astronaut space planet"

TOP 3 RECOMMENDATIONS :

1  Red Planet Rush
    A crew of astronauts races against time to establish the first human colony on Mars, facing dust storms, radiation, and equipment failures in space.
    Similarity Score: 0.3413

2  Galactic Frontier
    An astronaut and her robot companion explore uncharted planets in a distant galaxy, battling alien creatures and discovering ancient civilizations.
    Similarity Score: 0.2634

3  Cosmos Detective
    A sharp-witted detective solves crimes aboard a massive space station orbiting Saturn, using science and logic to catch interstellar criminals.
    Similarity Score: 0.1105
